# 📒 Notebook 2 — Integrasi Dataset & Resolusi Konflik
> **Peran:** Senior Data Scientist | **Proyek:** SPK Pantau Pasar Sumbawa

---

## 🔀 1. Strategi Integrasi Data

Empat sumber data yang telah dibersihkan akan digabungkan menggunakan `pd.concat()`. Karena beberapa sumber mencatat harga untuk tanggal dan komoditas yang **sama**, diperlukan mekanisme resolusi konflik.

### Hierarki Prioritas Data

| Prioritas | Sumber | Alasan |
|-----------|--------|--------|
| **1 (Tertinggi)** | SP2KP | Dataset resmi anchor dari Kementerian Perdagangan |
| **2** | Diskoperindag | Data instansi dinas kabupaten, tercatat harian |
| **3** | UPT_Seketeng | Laporan unit pasar, frekuensi mingguan |
| **4 (Terendah)** | PIHPS | Portal harga umum, kurang spesifik per varian |

### Strategi Resolusi Konflik
Jika kombinasi **Tanggal + Komoditas** terdapat lebih dari satu baris:
1. Assign bobot prioritas ke kolom `Priority`.
2. Sort berdasarkan `['Tanggal', 'Komoditas', 'Priority']` (ascending).
3. `drop_duplicates(keep='first')` → hanya simpan baris dengan prioritas tertinggi.

---


In [1]:
import logging
from pathlib import Path
from typing import Dict
import pandas as pd

# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("Integration")

# ── Paths ─────────────────────────────────────────────────────────────────────
CLEANED_DIR: Path = Path("../processed_data/cleaned")
MERGED_DIR: Path  = Path("../processed_data/merged")
MERGED_DIR.mkdir(parents=True, exist_ok=True)


## 📥 2. Load Dataset Cleaned


In [2]:
logger.info("Memuat dataset cleaned …")

df_pihps = pd.read_csv(CLEANED_DIR / "pihps_clean.csv",
                       parse_dates=["Tanggal"])
df_sp2kp = pd.read_csv(CLEANED_DIR / "sp2kp_clean.csv",
                       parse_dates=["Tanggal"])
df_disko = pd.read_csv(CLEANED_DIR / "diskoperindag_clean.csv",
                       parse_dates=["Tanggal"])
df_upt   = pd.read_csv(CLEANED_DIR / "upt_clean.csv",
                       parse_dates=["Tanggal"])

for name, df in [("PIHPS", df_pihps), ("SP2KP", df_sp2kp),
                  ("Diskoperindag", df_disko), ("UPT", df_upt)]:
    logger.info("  %-15s  %s baris", name, f"{len(df):,}")


10:43:58 [INFO] Memuat dataset cleaned …
10:43:58 [INFO]   PIHPS            20,940 baris
10:43:58 [INFO]   SP2KP            24,445 baris
10:43:58 [INFO]   Diskoperindag    7,963 baris
10:43:58 [INFO]   UPT              601 baris


## 🔗 3. Penggabungan & Resolusi Konflik


In [3]:
# ── Priority Hierarchy ────────────────────────────────────────────────────────
PRIORITY_MAP: Dict[str, int] = {
    "SP2KP":        1,
    "Diskoperindag": 2,
    "UPT_Seketeng": 3,
    "PIHPS":        4,
}

# ── Concatenate all four sources ──────────────────────────────────────────────
merged: pd.DataFrame = pd.concat(
    [df_sp2kp, df_disko, df_upt, df_pihps], ignore_index=True
)
logger.info("Setelah concat: %s baris", f"{len(merged):,}")

# ── Assign priority weight ────────────────────────────────────────────────────
merged["Priority"] = merged["Sumber"].map(PRIORITY_MAP)

# ── Sort: Date ↑, Commodity ↑, Priority ↑ ────────────────────────────────────
merged.sort_values(["Tanggal", "Komoditas", "Priority"], inplace=True)

# ── Deduplicate: keep highest-priority row per (Tanggal, Komoditas) ───────────
merged_clean: pd.DataFrame = (
    merged.drop_duplicates(subset=["Tanggal", "Komoditas"], keep="first")
    .drop(columns=["Priority"])
    .reset_index(drop=True)
)

logger.info("Setelah deduplikasi: %s baris", f"{len(merged_clean):,}")


10:43:58 [INFO] Setelah concat: 53,949 baris
10:43:58 [INFO] Setelah deduplikasi: 39,601 baris


## 📊 4. Distribusi Sumber dalam Dataset Gabungan


In [4]:
# ── Source distribution ───────────────────────────────────────────────────────
dist = merged_clean["Sumber"].value_counts()
print("Distribusi sumber dalam dataset gabungan:")
print("-" * 40)
for src, cnt in dist.items():
    pct = 100 * cnt / len(merged_clean)
    bar = "█" * int(pct / 2)
    print(f"  {src:<18} {cnt:>6,}  ({pct:5.1f}%)  {bar}")

print()
print(f"Total baris  : {len(merged_clean):,}")
print(f"Rentang waktu: {merged_clean['Tanggal'].min().date()} → "
      f"{merged_clean['Tanggal'].max().date()}")
print(f"Komoditas    : {merged_clean['Komoditas'].nunique()} unik")

# ── Save output ───────────────────────────────────────────────────────────────
output_path: Path = MERGED_DIR / "dataset_all_merged.csv"
merged_clean.to_csv(output_path, index=False)
logger.info("✅ Dataset gabungan disimpan → %s", output_path.resolve())
merged_clean.head(10)


10:43:58 [INFO] ✅ Dataset gabungan disimpan → C:\Users\Saputra Budiman\Documents\Tugas Kuliah Semester Genap 2025-2026\Sistem Informasi Manajemen\Skripsi - Pantau Pasar\SPK SIM\processed_data\merged\dataset_all_merged.csv


Distribusi sumber dalam dataset gabungan:
----------------------------------------
  SP2KP              24,445  ( 61.7%)  ██████████████████████████████
  PIHPS              12,083  ( 30.5%)  ███████████████
  Diskoperindag       2,783  (  7.0%)  ███
  UPT_Seketeng          290  (  0.7%)  

Total baris  : 39,601
Rentang waktu: 2021-01-04 → 2026-11-10
Komoditas    : 57 unik


,Tanggal,Komoditas,Harga,Sumber
0,2021-01-04,bawang_merah,35000.0,PIHPS
1,2021-01-04,bawang_putih_honan,30000.0,PIHPS
2,2021-01-04,beras_medium,10000.0,PIHPS
3,2021-01-04,beras_premium,12000.0,PIHPS
4,2021-01-04,cabai_merah_besar,42500.0,PIHPS
5,2021-01-04,cabai_merah_keriting,60000.0,PIHPS
6,2021-01-04,cabai_rawit_merah,62500.0,PIHPS
7,2021-01-04,daging_ayam_ras,41000.0,PIHPS
8,2021-01-04,daging_sapi_paha_belakang,120000.0,PIHPS
9,2021-01-04,daging_sapi_paha_depan,120000.0,PIHPS
